In [1]:
!pip install langchain-community langchain-core langchain[aws] langchain-tavily langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.9/174.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 100.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstal

In [2]:
import os
from typing import Annotated, Literal, Any, Generator
import json
import ast
import re
from time import time
import sys

from typing_extensions import TypedDict

import boto3
import requests
from botocore.config import Config
from langchain_aws.llms.bedrock import BedrockLLM
from langchain.chat_models import BedrockChat
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain_core.messages import ToolMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition, create_react_agent
from langchain.tools import tool
from langgraph.types import Command

ImportError: cannot import name 'BedrockChat' from 'langchain.chat_models' (/usr/local/lib/python3.12/dist-packages/langchain/chat_models/__init__.py)

In [3]:
from google.colab import userdata

os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

tavily_tool = TavilySearch(max_results=2)

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1",
    config=Config(read_timeout=1024),
    aws_access_key_id=userdata.get("AWS_STEPHENHO_SERVERLESS_ACC_ACCESS_TOKEN"),
    aws_secret_access_key=userdata.get("AWS_STEPHENHO_SERVERLESS_ACC_SECRET_ACCESS_KEY")
)

llm = init_chat_model(
    "us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    # "anthropic.claude-3-5-sonnet-20241022-v2:0",
    model_provider="bedrock_converse",
    client=bedrock_runtime,
    config={"max_tokens": 4096, "temperature": 0.7, "top_p": 0.8}
)

TimeoutException: Requesting secret TAVILY_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
class Router(TypedDict):
    """Worker to route to next. If no workers needed, route to FINISH."""
    next: Literal["joker", "stock_information_retriever", "stock_correlation_retriever", "crash_predictor", "FINISH"]

class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str # next agent to call
    eda_images: list[str]

In [ ]:
TASK_ROUTER_PROMPT = """
You are a task router tasked with managing a conversation between the following workers:
                              "joker", "stock_information_retriever", "stock_correlation_retriever", "crash_predictor".

                              Given the following user request, respond with the worker to act next.
                              Each worker will perform a task and respond with their results and status.
                              Analyze the results carefully and decide which worker to call next accordingly.
                              When finished, respond with FINISH.
"""


def task_router_node(state: State) -> Command[Literal["joker", "stock_information_retriever", "stock_correlation_retriever", "crash_predictor", "__end__"]]:
  messages = [{"role": "system", "content": TASK_ROUTER_PROMPT},] + state["messages"]
  response = llm.with_structured_output(Router).invoke(messages)
  print(">>>>>>>")
  print(response)
  goto = response["next"]
  if goto == "FINISH":
      goto = END

  return Command(goto=goto, update={"next": goto})

In [ ]:
@tool
def give_a_joke() -> str:
    """Give a joke."""
    response = llm.invoke("Tell me a joke.")
    return json.dumps({"joke": response.content})

joker_agent = create_react_agent(
    llm, tools=[give_a_joke], prompt="""
    You are an entertainer who give a joke.
    You get one joke from the large language model and you return this joke back to the user.
    """
)

def joker_node(state: State) -> Command[Literal["__end__"]]:
    result = joker_agent.invoke(state)
    return Command(goto="__end__", update={"messages": result['messages']})

In [ ]:
@tool
def stock_information_retriever(symbol: str, startdate: str, enddate: str) -> str:
    """Compute information for a given stock with a given date range, with dates in the format YYYY-mm-dd."""
    payload = {'symbol': symbol, 'startdate': startdate, 'enddate': enddate}
    headers = {'Content-Type': 'application/json'}
    response = requests.request("GET", "https://olqgg01k23.execute-api.us-east-1.amazonaws.com/prod/FinportFastAPILambda/api/v0/estimate-symbol-info", headers=headers, params=payload)
    return response.text

stock_information_retriever_agent = create_react_agent(
    llm, tools=[stock_information_retriever], prompt="""
    You are a stock information retriever, computing information about a stock given its symbole and a date range.
    And you return the information back to the user.
    """
)

def stock_information_retriever_node(state: State) -> Command[Literal["__end__"]]:
    result = stock_information_retriever_agent.invoke(state)
    return Command(goto="__end__", update={"messages": result['messages']})

In [ ]:
@tool
def stock_correlation_retriever(symbol1: str, symbol2: str, startdate: str, enddate: str) -> str:
    """Compute the correlation between two stocks with a given date range, with dates in the format YYYY-mm-dd."""
    payload = {'symbol1': symbol1, 'symbol2': symbol2, 'startdate': startdate, 'enddate': enddate}
    headers = {'Content-Type': 'application/json'}
    response = requests.request("GET", "https://olqgg01k23.execute-api.us-east-1.amazonaws.com/prod/FinportFastAPILambda/api/v0/estimate-symbols-correlation", headers=headers, params=payload)
    return response.text

stock_correlation_retriever_agent = create_react_agent(
    llm, tools=[stock_correlation_retriever], prompt="""
    You are an analyst who computes the correlation between two stocks for a given date range.
    And you return the information back to the user.
    """
)

def stock_correlation_retriever_node(state: State) -> Command[Literal["__end__"]]:
    result = stock_correlation_retriever_agent.invoke(state)
    return Command(goto="__end__", update={"messages": result['messages']})

In [ ]:
@tool
def crash_predictor(symbol: str="^GSPC", startdate: str=None, enddate: str=None) -> str:
    """Compute the potential crash date for either a given stock or S&P 500.
    If the symbol is not given, it is assumed to be S&P 500 (^GSPC).
    If the date range is not given, it is assumed to be the last one year.
    And you return the information back to the user.
    """
    payload = {'symbol': symbol, 'startdate': startdate, 'enddate': enddate}
    headers = {'Content-Type': 'application/json'}
    response = requests.request("GET", "https://olqgg01k23.execute-api.us-east-1.amazonaws.com/prod/FinportFastAPILambda/api/v0/predict-crash", headers=headers, params=payload)
    return response.text

crash_predictor_agent = create_react_agent(
    llm, tools=[crash_predictor], prompt="""
    You are a scientist who computes or predict the potential crash date of either a stock or S&P 500.
    And you return the information back to the user.
    """
)

def crash_predictor_node(state: State) -> Command[Literal["__end__"]]:
    result = crash_predictor_agent.invoke(state)
    return Command(goto="__end__", update={"messages": result['messages']})

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("task_router", task_router_node)
graph_builder.add_node("joker", joker_node)
graph_builder.add_node("stock_information_retriever", stock_information_retriever_node)
graph_builder.add_node("stock_correlation_retriever", stock_correlation_retriever_node)
graph_builder.add_node("crash_predictor", crash_predictor_node)

graph_builder.add_edge(START, "task_router")
graph_builder.add_conditional_edges(
    "task_router",
    tools_condition,
    {
        "joker": "joker",
        "stock_information_retriever": "stock_information_retriever",
        "stock_correlation_retriever": "stock_correlation_retriever",
        "crash_predictor": "crash_predictor",
        "__end__": END
    }
)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image as PythonImage
from IPython.display import display

try:
    display(PythonImage(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
from IPython.display import Image as PythonImage
from IPython.display import display

try:
    display(PythonImage(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            yield value

In [ ]:
def output_conversational_stream_texts(input_text: str) -> Generator[str, None, None]:
    starttime = time()
    for value in stream_graph_updates(input_text):
        print("======================", file=sys.stderr)
        current_time = time()
        print(f"Time elapsed: {current_time - starttime} sec", file=sys.stderr)
        print(value, file=sys.stderr)
        if 'messages' in value:
            for message in value['messages']:
                print(message, file=sys.stderr)
                print(type(message), file=sys.stderr)
                if isinstance(message, AIMessage):
                    if isinstance(message.content, str):
                        yield message.content
                    elif isinstance(message.content, list):
                        for content in message.content:
                            if content['type'] == 'text':
                                yield "-->"+content['text']
                            elif content['type'] == 'image_url':
                                yield f"Image URL: {content['image_url']}"
        print("****************")

In [ ]:
for output_str in output_conversational_stream_texts("Compute the information for GOOG from 2020 up to present."):
    print(output_str)

>>>>>>>
{'next': 'stock_information_retriever'}
****************


Time elapsed: 1.5680882930755615 sec
{'next': 'stock_information_retriever'}


-->I'll help you retrieve the stock information for Google (GOOG) from 2020 to present. Let me make that function call for you.
Based on the analysis of Google (GOOG) stock from January 1, 2020, to January 24, 2024, here are the key findings:

1. Return (r): 27.71% - This represents the total return over the entire period
2. Volatility: 37.25% - This measures the degree of variation in the stock price
3. Risk Metrics:
   - Downside Risk: 26.06%
   - Upside Risk: 26.66%
4. Beta: 1.139 - This indicates that GOOG is slightly more volatile than the market (Beta > 1)
5. Data Coverage: The analysis includes 1,022 trading days, from January 2, 2020, to January 24, 2024

The beta of 1.139 suggests that GOOG tends to move in the same direction as the market but with slightly higher magnitude. The relatively balanced upside and downside risks indicate symmetric volatility in both directions.
****************


Time elapsed: 41.808594942092896 sec
{'messages': [HumanMessage(content='Compute the information for GOOG from 2020 up to present.', additional_kwargs={}, response_metadata={}, id='6446f38b-4522-4279-b199-b08fb6c11ba4'), AIMessage(content=[{'type': 'text', 'text': "I'll help you retrieve the stock information for Google (GOOG) from 2020 to present. Let me make that function call for you."}, {'type': 'tool_use', 'name': 'stock_information_retriever', 'input': {'symbol': 'GOOG', 'startdate': '2020-01-01', 'enddate': '2024-01-24'}, 'id': 'tooluse_VkfM7Yd5RJavax9FiQ1hvw'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '621ef446-9da7-4256-9f54-d0275c337f8b', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 24 Jul 2025 02:20:33 GMT', 'content-type': 'application/json', 'content-length': '479', 'connection': 'keep-alive', 'x-amzn-requestid': '621ef446-9da7-4256-9f54-d0275c337f8b'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [5682]